# HECS Free AI Video Studio (Wan 2.2 on free Google Colab)

Make AI video clips **for free** with an open-source model, then join them, add music, captions and your logo, and export the final video.

| | |
|---|---|
| **Model** | Wan 2.2 TI2V-5B by Alibaba Wan team. License: **Apache-2.0** (commercial use allowed) |
| **Engine** | ComfyUI, run in the background. You only use this notebook. No extra website is needed. |
| **Does** | Text → video, Image → video, continuing a scene, 16:9 (YouTube) and 9:16 (Reels/Shorts) |
| **Output** | 1280×704 or 704×1280 at 24 fps, 1–5 seconds per clip. The editor at the end exports 1080p MP4 (upscaled). |
| **Cost** | ₹0. Uses Colab's free GPU. Google doesn't guarantee free GPU time, and daily limits vary. |

### How to use (first time)
1. **Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save**.
2. Run cells **① → ④** in order: click ▶ on each, wait for "OK".
3. Fill in the form in cell **⑤** and run it to make a clip. Run ⑤ again for each new clip.
4. Use **⑥** to download, and **⑦** to join clips and add music, captions and your logo.

> **Free-tier rules (from Google's Colab FAQ):** stay on this tab while it works. Don't use tunnels or remote web UIs, because Colab may stop free sessions that do. A free session lasts at most 12 hours and ends sooner if idle. **Download your videos before closing**, because everything in `/content` is erased.

In [ ]:
#@title ① Check the free GPU (run this first)
import subprocess, shutil, sys, os
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    raise SystemExit("NO GPU FOUND.  Go to  Runtime > Change runtime type > T4 GPU > Save,  then run this cell again.\n"
                     "If Colab says no GPU is available, the free quota is used up for now - try again later (often a few hours).")
import psutil, torch
print("OK  GPU      :", r.stdout.strip())
print(f"OK  RAM      : {psutil.virtual_memory().total/1e9:.1f} GB")
print(f"OK  Disk free: {shutil.disk_usage('/content').free/1e9:.0f} GB  (need about 25 GB)")
print(f"OK  PyTorch  : {torch.__version__}  CUDA {torch.version.cuda}  Python {sys.version.split()[0]}")
if shutil.disk_usage('/content').free < 25e9:
    print("WARNING: less than 25 GB free disk - the download may fail.")

In [ ]:
#@title ② Install the ComfyUI engine (about 2–4 min)
COMFY_TAG = "v0.45.4"   # pinned, tested version (15 Nov 2026). Change only if you know why.
import subprocess, os, sys

def run(cmd, cwd=None):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-3000:]); print(p.stderr[-3000:])
        raise SystemExit(f"FAILED: {cmd}")
    return p.stdout

if not os.path.exists("/content/ComfyUI/main.py"):
    run(f"git clone --depth 1 --branch {COMFY_TAG} https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI")
import torch
torch_before = torch.__version__
run(f"{sys.executable} -m pip install -q -r requirements.txt", cwd="/content/ComfyUI")
run("apt-get -qq install -y ffmpeg > /dev/null 2>&1 || true")
torch_after = run(f"{sys.executable} -c 'import torch;print(torch.__version__)'").strip()
print("OK  ComfyUI", COMFY_TAG, "installed")
print("OK  PyTorch", torch_after, "(was", torch_before + ")")
if torch_after != torch_before:
    print("NOTE: pip changed PyTorch. Use  Runtime > Restart session, then continue from cell ③ (no need to re-run ②).")

In [ ]:
#@title ③ Download the Wan 2.2 5B model files (about 17 GB, usually 3–8 min)
#@markdown Free public files from Hugging Face (no account or token needed).
import os, shutil, time
from huggingface_hub import hf_hub_download

FILES = [  # (repo, file inside repo, ComfyUI folder, approx size GB)
  ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors", "diffusion_models", 10.0),
  ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/vae/wan2.2_vae.safetensors",                     "vae",              1.4),
  ("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "text_encoders", 6.7),
]
t0 = time.time()
for repo, path, folder, gb in FILES:
    name = os.path.basename(path)
    dest = f"/content/ComfyUI/models/{folder}/{name}"
    if os.path.exists(dest) and os.path.getsize(dest) > 0.9 * gb * 1e9:
        print("already have", name); continue
    print(f"downloading {name} (~{gb} GB) ...", flush=True)
    p = hf_hub_download(repo_id=repo, filename=path, local_dir="/content/hf_download")
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.move(p, dest)
    print(f"   OK {os.path.getsize(dest)/1e9:.2f} GB")
shutil.rmtree("/content/hf_download", ignore_errors=True)
print(f"OK  all model files ready in {(time.time()-t0)/60:.1f} min")

In [ ]:
#@title ④ Start the video engine (runs in the background, about 30 s)
import subprocess, time, json, urllib.request, sys, os
URL = "http://127.0.0.1:8188"

def api(path, data=None, timeout=30):
    req = urllib.request.Request(URL + path, data=None if data is None else json.dumps(data).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read() or b"{}")

def engine_ready(timeout=5):
    try: api("/system_stats", timeout=timeout); return True
    except Exception: return False

def engine_alive():
    # The engine is busy (not dead) while it loads 17 GB of models, so check the process first.
    if "ENGINE" in globals():
        return ENGINE.poll() is None
    return engine_ready(timeout=60)

if not engine_ready():
    LOG = open("/content/comfyui.log", "w")
    ENGINE = subprocess.Popen([sys.executable, "-u", "main.py", "--listen", "127.0.0.1", "--port", "8188",
                               "--preview-method", "none"],
                              cwd="/content/ComfyUI", stdout=LOG, stderr=subprocess.STDOUT,
                              env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for _ in range(120):
        time.sleep(2)
        if engine_ready(): break
        if ENGINE.poll() is not None: break
if not engine_ready(timeout=30):
    print(open("/content/comfyui.log").read()[-4000:])
    raise SystemExit("Engine did not start - see the log above (Troubleshooting section at the bottom).")
s = api("/system_stats")
dev = s["devices"][0]
print(f"OK  engine running  |  ComfyUI {s['system'].get('comfyui_version')}  |  {dev['name']}  |  VRAM free {dev['vram_free']/1e9:.1f} GB")

## ⑤ Make a clip
Write what you want in the form below, then run the cell. **Each run makes one clip** and saves it to `/content/outputs`.

**Prompt formula:** *shot type + subject + action + setting + lighting + camera movement + style*.
For example: *"Cinematic close-up of a matte-black wireless earbud case opening on a wet black marble surface, soft rim light, slow dolly-in, shallow depth of field, premium commercial look."*

| Setting | Tip |
|---|---|
| **Seconds** | Start with 2–3 s. Measured on the free T4: a 3 s, 720p clip at Standard quality takes about 16 min (≈37 s per step). 5 s takes roughly twice as long. |
| **Quality** | *Draft* (12 steps, ~40% faster) = preview. *Standard* (20 steps) = normal. *High* (30 steps) = slower and slightly cleaner. |
| **Seed** | `-1` = random. Reuse the printed seed to re-make the same shot with a small change. |
| **Image → video** | Upload a product photo or a key-frame. The clip starts from that picture, which keeps products and characters consistent. |
| **Continue last clip** | Starts from the final frame of your previous clip, so it continues the scene. |

In [ ]:
#@title ⑤ Generate a clip
MODE = "Text to video" #@param ["Text to video", "Image to video (upload a picture)", "Continue last clip"]
PROMPT = "Cinematic slow dolly-in on a glass bottle of premium cold-pressed mango juice standing on a wooden table, morning sunlight through a window, soft shadows, condensation droplets on the glass, shallow depth of field, photorealistic commercial look" #@param {type:"string"}
NEGATIVE = "blurry, low quality, distorted, deformed hands, extra fingers, text, watermark, logo, flicker, jitter, static frame, oversaturated, cartoon" #@param {type:"string"}
FORMAT = "16:9 Landscape 1280x704 (YouTube / web)" #@param ["16:9 Landscape 1280x704 (YouTube / web)", "9:16 Vertical 704x1280 (Reels / Shorts)", "1:1 Square 960x960 (feed, experimental)"]
SECONDS = 3 #@param {type:"slider", min:1, max:5, step:1}
QUALITY = "Standard (20 steps)" #@param ["Draft (12 steps)", "Standard (20 steps)", "High (30 steps)"]
SEED = -1 #@param {type:"integer"}
LOW_MEMORY_MODE = False #@param {type:"boolean"}
#@markdown *Low memory mode stores the model in 8-bit (fp8). Turn it on only if you get "out of memory" errors. Quality drops slightly.*

import os, json, time, random, shutil, glob, subprocess, re, uuid, urllib.error
from IPython.display import display, Video, clear_output
from PIL import Image, ImageOps

assert 'engine_alive' in globals() and engine_alive(), "Run cell ④ first (engine not running)."
W, H = (1280, 704) if FORMAT.startswith("16:9") else (704, 1280) if FORMAT.startswith("9:16") else (960, 960)
FPS = 24
LENGTH = FPS * SECONDS + 1                       # Wan needs 4n+1 frames: 25, 49, 73, 97, 121
STEPS = int(re.search(r"(\d+) steps", QUALITY).group(1))
seed = random.randint(0, 2**32 - 1) if SEED < 0 else SEED
OUT = "/content/outputs"; os.makedirs(OUT, exist_ok=True)
clip_no = len(glob.glob(f"{OUT}/clip_*.mp4")) + 1
prefix = f"hecs_{clip_no:03d}_{uuid.uuid4().hex[:6]}"

def fit_image(src, dst, w, h):
    im = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
    ImageOps.fit(im, (w, h), Image.LANCZOS, centering=(0.5, 0.5)).save(dst)

start_image = None
if MODE.startswith("Image"):
    from google.colab import files
    print("Choose a picture (JPG/PNG). It will be centre-cropped to", W, "x", H)
    up = files.upload()
    if not up: raise SystemExit("No picture uploaded.")
    start_image = f"{prefix}_start.png"
    fit_image(list(up.keys())[0], f"/content/ComfyUI/input/{start_image}", W, H)
elif MODE.startswith("Continue"):
    lasts = sorted(glob.glob(f"{OUT}/clip_*_lastframe.png"))
    if not lasts: raise SystemExit("No previous clip found - make a clip first.")
    start_image = f"{prefix}_start.png"
    fit_image(lasts[-1], f"/content/ComfyUI/input/{start_image}", W, H)
    print("Continuing from", os.path.basename(lasts[-1]))

graph = {
  "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "wan2.2_ti2v_5B_fp16.safetensors",
                                               "weight_dtype": "fp8_e4m3fn" if LOW_MEMORY_MODE else "default"}},
  "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan", "device": "default"}},
  "3": {"class_type": "VAELoader", "inputs": {"vae_name": "wan2.2_vae.safetensors"}},
  "4": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["1", 0], "shift": 8.0}},
  "5": {"class_type": "CLIPTextEncode", "inputs": {"text": PROMPT, "clip": ["2", 0]}},
  "6": {"class_type": "CLIPTextEncode", "inputs": {"text": NEGATIVE, "clip": ["2", 0]}},
  "7": {"class_type": "Wan22ImageToVideoLatent", "inputs": {"vae": ["3", 0], "width": W, "height": H, "length": LENGTH, "batch_size": 1}},
  "8": {"class_type": "KSampler", "inputs": {"model": ["4", 0], "seed": seed, "steps": STEPS, "cfg": 5.0,
                                             "sampler_name": "uni_pc", "scheduler": "simple",
                                             "positive": ["5", 0], "negative": ["6", 0], "latent_image": ["7", 0], "denoise": 1.0}},
  "9": {"class_type": "VAEDecode", "inputs": {"samples": ["8", 0], "vae": ["3", 0]}},
  "10": {"class_type": "SaveImage", "inputs": {"images": ["9", 0], "filename_prefix": prefix}},
}
if start_image:
    graph["11"] = {"class_type": "LoadImage", "inputs": {"image": start_image}}
    graph["7"]["inputs"]["start_image"] = ["11", 0]

try:
    pid = api("/prompt", {"prompt": graph, "client_id": "hecs-notebook"})["prompt_id"]
except urllib.error.HTTPError as e:
    raise SystemExit("Engine rejected the job:\n" + e.read().decode()[:3000])

print(f"Clip {clip_no}: {W}x{H}, {SECONDS}s ({LENGTH} frames), {STEPS} steps, seed {seed}")
print("Working... first run also loads the model (1–3 min). A 3 s Standard clip takes ~16 min on a free T4. Keep this tab open.")
t0 = time.time(); last = ""
while True:
    time.sleep(10)
    if not engine_alive():
        print(open("/content/comfyui.log").read()[-3000:])
        raise SystemExit("The engine stopped (often: out of RAM). Re-run cell ④, enable LOW_MEMORY_MODE, try fewer seconds.")
    try:
        h = api(f"/history/{pid}", timeout=60)
    except Exception:
        continue                                   # engine busy loading models - keep waiting
    if pid in h:
        st = h[pid].get("status", {})
        if st.get("status_str") == "error":
            msgs = [m for m in st.get("messages", []) if m[0] == "execution_error"]
            print(json.dumps(msgs, indent=1)[:3000])
            raise SystemExit("Generation failed - see the error above and the Troubleshooting section.")
        break
    log_tail = open("/content/comfyui.log", errors="ignore").read()[-2000:]
    prog = re.findall(r"(\d+)/(\d+) \[[^\]]*\]", log_tail)
    msg = f"  {int(time.time()-t0)//60} min {int(time.time()-t0)%60:02d} s elapsed" + (f"  |  step {prog[-1][0]}/{prog[-1][1]}" if prog else "")
    if msg != last: print(msg, flush=True); last = msg

imgs = [i for node in h[pid]["outputs"].values() for i in node.get("images", [])]
frames = sorted(os.path.join("/content/ComfyUI/output", i.get("subfolder", ""), i["filename"]) for i in imgs)
if not frames: raise SystemExit("The engine finished but returned no frames - check /content/comfyui.log")
fdir = f"/content/frames/{prefix}"; os.makedirs(fdir, exist_ok=True)
skip_first = 1 if MODE.startswith("Continue") else 0      # avoid a repeated frame when continuing a scene
for n, f in enumerate(frames[skip_first:]):
    shutil.copy(f, f"{fdir}/f_{n:05d}.png")
mp4 = f"{OUT}/clip_{clip_no:03d}.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-framerate", str(FPS), "-i", f"{fdir}/f_%05d.png",
                "-c:v", "libx264", "-preset", "slow", "-crf", "16", "-pix_fmt", "yuv420p", "-movflags", "+faststart", mp4], check=True)
shutil.copy(frames[-1], f"{OUT}/clip_{clip_no:03d}_lastframe.png")
for f in frames: os.remove(f)
json.dump({"clip": clip_no, "mode": MODE, "prompt": PROMPT, "negative": NEGATIVE, "width": W, "height": H,
           "seconds": SECONDS, "frames": LENGTH, "steps": STEPS, "cfg": 5.0, "seed": seed, "low_memory": LOW_MEMORY_MODE,
           "model": "Wan2.2-TI2V-5B (Apache-2.0)", "minutes": round((time.time()-t0)/60, 1)},
          open(f"{OUT}/clip_{clip_no:03d}.json", "w"), indent=1)
print(f"DONE in {(time.time()-t0)/60:.1f} min  ->  {mp4}   (seed {seed})")
display(Video(mp4, embed=True, width=640 if W > H else 320))

In [ ]:
#@title ⑥ Download clips / save to Google Drive
ACTION = "Download all clips as a ZIP" #@param ["Download all clips as a ZIP", "Download the latest clip", "Copy everything to Google Drive"]
import glob, os, shutil
from google.colab import files
OUT = "/content/outputs"
if ACTION.startswith("Download all"):
    shutil.make_archive("/content/hecs_clips", "zip", OUT); files.download("/content/hecs_clips.zip")
elif ACTION.startswith("Download the latest"):
    files.download(sorted(glob.glob(f"{OUT}/*.mp4"))[-1])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = "/content/drive/MyDrive/HECS_AI_Video"; os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f"{OUT}/*"): shutil.copy(f, dst)
    print("Copied to Google Drive ▸ My Drive ▸ HECS_AI_Video")

## ⑦ Finish the video: join clips, add music, captions and logo, export 1080p
Everything here uses **FFmpeg**, which is free and open source. Run the cell. It asks you to upload optional music, a logo or captions.

* **Music:** use only audio you have the rights to. Good sources are the YouTube Audio Library and Pixabay Music. Check each track's licence.
* **Captions:** a `.srt` file. You can write one in Notepad, as in the example in the guide.
* **1080p:** the clips are 720p. This step upscales them to 1920×1080 or 1080×1920, which platforms accept. It doesn't add new detail.

In [ ]:
#@title ⑦ Edit & export the final video
CLIPS = "all" #@param {type:"string"}
#@markdown `all`, or clip numbers in order, e.g. `1,2,4`
TRANSITION = "Crossfade 0.5 s" #@param ["Hard cut", "Crossfade 0.5 s"]
ADD_MUSIC = True #@param {type:"boolean"}
MUSIC_VOLUME = 0.8 #@param {type:"slider", min:0.1, max:1.0, step:0.1}
ADD_LOGO = False #@param {type:"boolean"}
LOGO_POSITION = "top-right" #@param ["top-right", "top-left", "bottom-right", "bottom-left"]
ADD_CAPTIONS_SRT = False #@param {type:"boolean"}
EXPORT = "1080p (upscaled)" #@param ["1080p (upscaled)", "Original 720p"]
OUTPUT_NAME = "HECS_final" #@param {type:"string"}

import glob, os, subprocess, json, re, shutil
from google.colab import files
from IPython.display import display, Video
OUT = "/content/outputs"
allc = sorted(glob.glob(f"{OUT}/clip_[0-9][0-9][0-9].mp4"))
if CLIPS.strip().lower() != "all":
    allc = [f"{OUT}/clip_{int(n):03d}.mp4" for n in re.findall(r"\d+", CLIPS)]
if not allc: raise SystemExit("No clips found - make clips with cell ⑤ first.")

def probe(f):
    j = json.loads(subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=width,height:format=duration",
                                   "-of", "json", f], capture_output=True, text=True).stdout)
    return j["streams"][0]["width"], j["streams"][0]["height"], float(j["format"]["duration"])
W, H, _ = probe(allc[0])
sizes = {probe(c)[:2] for c in allc}
if len(sizes) > 1: raise SystemExit(f"Clips have different sizes {sizes}. Join only clips of the same format (all 16:9 or all 9:16).")
durs = [probe(c)[2] for c in allc]

def ask(kind, exts):
    print(f"Upload your {kind} ({exts}) ..."); up = files.upload()
    return os.path.abspath(list(up.keys())[0]) if up else None
music = ask("music", "mp3/wav/m4a") if ADD_MUSIC else None
logo = ask("logo", "PNG with transparent background") if ADD_LOGO else None
srt = ask("captions", ".srt") if ADD_CAPTIONS_SRT else None

inputs = sum([["-i", c] for c in allc], [])
fc, v = [], "[0:v]"
for i in range(len(allc)): fc.append(f"[{i}:v]fps=24,scale={W}:{H},setsar=1,format=yuv420p[v{i}]")
if len(allc) == 1:
    v = "[v0]"; total = durs[0]
elif TRANSITION.startswith("Cross"):
    d, off, prev = 0.5, 0.0, "[v0]"
    for i in range(1, len(allc)):
        off += durs[i-1] - d
        fc.append(f"{prev}[v{i}]xfade=transition=fade:duration={d}:offset={off:.3f}[x{i}]"); prev = f"[x{i}]"
    v = prev; total = sum(durs) - d * (len(allc) - 1)
else:
    fc.append("".join(f"[v{i}]" for i in range(len(allc))) + f"concat=n={len(allc)}:v=1:a=0[cat]"); v = "[cat]"; total = sum(durs)

FW = W
if EXPORT.startswith("1080"):
    if W > H:   fc.append(f"{v}scale=-2:1080:flags=lanczos,crop=1920:1080[sc]"); FW = 1920
    elif H > W: fc.append(f"{v}scale=1080:-2:flags=lanczos,crop=1080:1920[sc]"); FW = 1080
    else:       fc.append(f"{v}scale=1080:1080:flags=lanczos[sc]"); FW = 1080
    v = "[sc]"
n = len(allc)
if logo:
    inputs += ["-i", logo]
    pos = {"top-right": "W-w-40:40", "top-left": "40:40", "bottom-right": "W-w-40:H-h-40", "bottom-left": "40:H-h-40"}[LOGO_POSITION]
    fc.append(f"[{n}:v]format=rgba,scale={int(FW*0.14)//2*2}:-1[lg]")
    fc.append(f"{v}[lg]overlay={pos}[lo]"); v = "[lo]"; n += 1
if srt:
    shutil.copy(srt, "/content/captions.srt")
    fc.append(f"{v}subtitles=/content/captions.srt:force_style='FontName=DejaVu Sans,FontSize=14,Outline=2,MarginV=40'[sb]"); v = "[sb]"
audio_map = []
if music:
    inputs += ["-i", music]
    fade = max(total - 2, 0)
    fc.append(f"[{n}:a]volume={MUSIC_VOLUME},afade=t=in:d=0.5,afade=t=out:st={fade:.2f}:d=2,atrim=0:{total:.2f}[aout]")
    audio_map = ["-map", "[aout]", "-c:a", "aac", "-b:a", "192k"]
final = f"{OUT}/{OUTPUT_NAME}.mp4"
cmd = ["ffmpeg", "-y", "-loglevel", "error", *inputs, "-filter_complex", ";".join(fc), "-map", v, *audio_map,
       "-c:v", "libx264", "-preset", "slow", "-crf", "18", "-pix_fmt", "yuv420p", "-r", "24",
       "-t", f"{total:.2f}", "-movflags", "+faststart", final]
p = subprocess.run(cmd, capture_output=True, text=True)
if p.returncode != 0: print(p.stderr[-3000:]); raise SystemExit("FFmpeg failed - see above.")
fw, fh, fd = probe(final)
print(f"DONE  {final}  |  {fw}x{fh}  |  {fd:.1f} s  |  {os.path.getsize(final)/1e6:.1f} MB")
display(Video(final, embed=True, width=640 if fw > fh else 320))
files.download(final)

## Troubleshooting

| Problem | What to do |
|---|---|
| **"NO GPU FOUND" / "cannot connect to GPU backend"** | Runtime ▸ Change runtime type ▸ T4 GPU. If Colab refuses, the free quota is used up. Wait a few hours or try the next day. |
| **Session crashed / engine stopped while generating** | Usually system RAM. Re-run ④ (and ③ if files are missing), tick **LOW_MEMORY_MODE**, and use 2–3 seconds. |
| **"CUDA out of memory"** | Tick **LOW_MEMORY_MODE**, lower **Seconds**, or use Draft quality. Runtime ▸ Restart session clears the GPU. |
| **Very slow** | Expected on a free T4 (~37 s per step at 3 s / 720p). Use 2–3 s clips and Draft quality to test prompts, then re-make the best one in Standard/High with the same seed. |
| **Download error in ③** | Re-run ③. It skips files that are already complete. If Hugging Face is down, try later. |
| **Everything disappeared** | Colab wipes `/content` when the session ends. Re-run ①–④ (≈10 min). Always download or save to Drive (⑥). |
| **Clip looks too static** | The 5B model plays motion safe. Name a clear action and camera move ("camera orbits around", "liquid pours into the glass", "steam rises"). Avoid only-camera prompts like "slow dolly-in" on a still object. |
| **Text/logos in the video look wrong** | AI video can't spell reliably. Add text and logos in step ⑦, not in the prompt. |
| **Engine log** | Open `/content/comfyui.log` in the Files panel (left sidebar). |

**Licence reminders:** Wan 2.2 is Apache-2.0, so commercial use is allowed. ComfyUI is GPL-3.0, and you are only running it, not redistributing it. Use music, logos, faces and brand assets only if you have the rights to them.